# Dependancies 

In [ ]:
import os
from openpyxl import Workbook
import matplotlib.pyplot as plt
import pandas as pd

# Something

In [ ]:
import ME_modules.MakeWriteExcel as MWE
import ME_modules.Folder_n_File_Utilities as fnfu
import  ME_modules.ME_austin_EIS.chip_processor as MCP

# Main

In [ ]:
def main(use_hardcoded_path=False, hardcoded_path=None):
    # 1. Setup Phase
    # Create the initial workbooks structure
    processed_chips_folder = "Processed Chips for PCAI1ia"
    MWE.create_chips_folder_and_workbooks()
    
    # Determine the master folder path
    if use_hardcoded_path and hardcoded_path:
        master_folder = hardcoded_path
        print(f"Using hardcoded path: {master_folder}")
    else:
        master_folder = fnfu.select_folder()
        if not master_folder:
            print("No folder selected. Exiting.")
            return

    # Set up output folder and logging
    output_folder = os.path.join(master_folder, "Processed_Chip_Data")
    os.makedirs(output_folder, exist_ok=True)
    log_file = os.path.join(output_folder, "analysis_log.txt")
    open(log_file, 'w').close()

    # 2. Chip Validation Phase
    chip_data = fnfu.find_valid_chips(master_folder)
    valid_chips, invalid_chips = fnfu.match_chip_files(chip_data)
    
    fnfu.log_and_print("A valid chip must contain: water.csv, buffer.csv, dna.csv, cyst.csv, cas.xlsx", log_file)
    fnfu.log_and_print(f"Found {len(chip_data)} chip groups in all subfolders.", log_file)
    
    if not valid_chips:
        fnfu.log_and_print("No valid chips found.", log_file)
        return
    
    if invalid_chips:
        fnfu.log_and_print("Invalid chips found. Missing files:", log_file)
        for chip, missing in sorted(invalid_chips.items(), key=lambda x: int(x[0])):
            missing_types = [key for key in ["water", "buffer", "dna", "cyst", "cas"] if key not in missing]
            fnfu.log_and_print(f"Chip {chip} is missing: {', '.join(missing_types)}", log_file)

    # 3. Chip Selection Phase
    fnfu.log_and_print(f"Valid chips found: {sorted(valid_chips.keys(), key=int)}", log_file)
    chip_choice = input("Enter chip number to process or 'all' to process all valid chips: ").strip()
    
    chips_to_process = []
    if chip_choice.lower() == 'all':
        chips_to_process = sorted(valid_chips.keys(), key=int)
    elif chip_choice in valid_chips:
        chips_to_process = [chip_choice]
    else:
        fnfu.log_and_print(f"Invalid selection: {chip_choice}", log_file)
        return

    # 4. Processing Phase
    for chip in chips_to_process:
        chip_paths = valid_chips[chip]
        fnfu.log_and_print(f"\nProcessing Chip {chip}...", log_file)
        
        try:
            # Process the Excel file for the chip
            chip_results = MCP.process_chip_excel_only(chip, chip_paths)

            if not chip_results:
                fnfu.log_and_print(f"No valid results for Chip {chip}", log_file)
                continue

            # Write results into correct workbook/sheets
            MCP.write_chip_results_to_workbook(chip_results, processed_chips_folder)
            fnfu.log_and_print(f"Successfully processed and wrote results for Chip {chip}", log_file)
            
        except Exception as e:
            fnfu.log_and_print(f"Error processing Chip {chip}: {str(e)}", log_file)
            continue
    
    print(f"MCP.all_raw_names_logged:{MCP.all_raw_names_logged}")
    fnfu.log_and_print("\nProcessing completed!", log_file)

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
import os
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define your lists
workbook_names = [ 'Chip 21', 'Chip 22', 'Chip 23', 'Chip 26', 'Chip 27', 'Chip 32', 'Chip 33', 'Chip 35', 'Chip 36', 'Chip 37', 'Chip 39', 'Chip 40', 'Chip 41', 'Chip 43', 'Chip 44', 'Chip 45', 'Chip 46', 'Chip 47', 'Chip 48', 'Chip 52', 'Chip 53', 'Chip 54', 'Chip 55', 'Chip 56', 'Chip 57', 'Chip 59' ]
worksheet_names = ["0pM_asso", "0pM_disso", "100pM_asso", "100pM_disso", "1nM_asso", "1nM_disso", "10nM_asso", "10nM_disso", "100nM_asso", "100nM_disso"]
headers = [ 'time(s)', 'delta Rct-a', 'Rct-d', 'Cp1', 'Ph1', 'Slope 1', 'Slope 2', 'Slope 3', 'Slope 4', 'Slope 5', 'Angle', 'Cp_exp-a', 'Cp_exp-b', 'Ph_slope', 'Ph_peak', 'Area Cp', 'Area Ph', 'Area Slope', 'Area Rs-direct', 'Area Rs-Para', '','','linear_eq_slope', 'linear_eq_b', 'Rs', 'delta Rct-i', 'Q', 'n']

def select_folder():
    root = tk.Tk()
    root.withdraw()
    return filedialog.askdirectory(title="Select folder containing workbooks")

def read_workbook_data(folder_path):
    data = {}  # {worksheet: {header: [values across chips]}}

    for chip_name in workbook_names:
        file_path = os.path.join(folder_path, f"{chip_name}.xlsx")
        if not os.path.exists(file_path):
            print(f"Workbook not found: {file_path}")
            continue

        wb = pd.ExcelFile(file_path)
        for ws in worksheet_names:
            if ws not in wb.sheet_names:
                continue
            df = wb.parse(ws)
            if ws not in data:
                data[ws] = {h: [] for h in headers}
            for h in headers:
                if h in df.columns:
                    data[ws][h].append(df[h].dropna().values)
                else:
                    data[ws][h].append([])

    return data

def plot_and_save(data, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Subfolder names based on worksheet_names
    worksheet_folders = {ws: os.path.join(output_folder, f"{ws}_graphs") for ws in worksheet_names}
    for folder in worksheet_folders.values():
        os.makedirs(folder, exist_ok=True)

    for ws, header_data in data.items():
        for h, all_values in header_data.items():
            # Prepare DataFrame
            combined = []
            for chip_name, vals in zip(workbook_names, all_values):
                # Filter out empty or non-numeric values
                numeric_vals = []
                for v in vals:
                    try:
                        numeric_vals.append(float(v))
                    except (ValueError, TypeError):
                        continue
                if len(numeric_vals) > 0:
                    combined.append(pd.DataFrame({'Chip': chip_name, 'Value': numeric_vals}))

            if not combined:
                continue

            df = pd.concat(combined, ignore_index=True)

            if df['Value'].dropna().empty:
                continue

            # Prepare data for boxplot
            data_for_plot = []
            chip_labels = []
            for chip in df['Chip'].unique():
                values = df[df['Chip'] == chip]['Value'].dropna().astype(float)
                if len(values) > 0:
                    data_for_plot.append(values)
                    chip_labels.append(chip)

            if len(data_for_plot) < 2:
                continue

            # Plot
            plt.figure(figsize=(10, 6))
            plt.title(f"{ws} - {h}")
            plt.ylabel(h)
            plt.xlabel("Chip")
            plt.xticks(rotation=90)
            plt.grid(True, linestyle="--", alpha=0.5)
            plt.boxplot(data_for_plot, tick_labels=chip_labels)
            plt.tight_layout()

            # Save in correct subfolder
            safe_name = f"{ws}_{h}".replace('/', '-').replace('\\', '-')
            save_path = os.path.join(worksheet_folders[ws], f"{safe_name}.png")
            plt.savefig(save_path)
            plt.close()
            print(f"Saved plot: {save_path}")

def plot_vs_time(data, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Subfolder for time-based plots
    time_plots_folder = os.path.join(output_folder, "time_vs_column_graphs")
    os.makedirs(time_plots_folder, exist_ok=True)

    for ws, header_data in data.items():
        if 'time(s)' not in header_data:
            continue

        time_values_per_chip = header_data['time(s)']

        for h in headers:
            if h == 'time(s)':
                continue

            plt.figure(figsize=(10, 6))
            plt.title(f"{ws} - {h} vs. time(s)")
            plt.xlabel("time(s)")
            plt.ylabel(h)
            plt.grid(True, linestyle="--", alpha=0.5)

            any_data = False

            for chip_name, time_vals, y_vals in zip(workbook_names, time_values_per_chip, header_data[h]):
                if len(time_vals) == 0 or len(y_vals) == 0:
                    continue

                try:
                    time_numeric = np.array(time_vals, dtype=float)
                    y_numeric = np.array(y_vals, dtype=float)
                except:
                    continue

                if len(time_numeric) != len(y_numeric):
                    continue

                plt.plot(time_numeric, y_numeric, label=chip_name)
                any_data = True

            if not any_data:
                plt.close()
                continue

            plt.legend(fontsize=8)
            plt.tight_layout()

            safe_name = f"{ws}_{h}_vs_time".replace('/', '-').replace('\\', '-')
            save_path = os.path.join(time_plots_folder, f"{safe_name}.png")
            plt.savefig(save_path)
            plt.close()
            print(f"Saved time-based plot: {save_path}")



def comparitive_graphs():
    folder_path = select_folder()
    if not folder_path:
        print("No folder selected. Exiting.")
        return

    output_folder = os.path.abspath("Processed Chips for PCAI1ia")
    data = read_workbook_data(folder_path)
    plot_and_save(data, output_folder)
    plot_vs_time(data, output_folder)  # Add this line
    print(f"Plots saved in {output_folder}")


comparitive_graphs()


C:\Users\austi\AppData\Local\Temp\ipykernel_27728\3985480093.py:98: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_for_plot, labels=chip_labels)


Saved plot: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\0pM_asso_graphs\0pM_asso_time(s).png


C:\Users\austi\AppData\Local\Temp\ipykernel_27728\3985480093.py:98: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_for_plot, labels=chip_labels)


Saved plot: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\0pM_asso_graphs\0pM_asso_delta Rct-a.png


C:\Users\austi\AppData\Local\Temp\ipykernel_27728\3985480093.py:98: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_for_plot, labels=chip_labels)


...

Saved plot: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\100nM_disso_graphs\100nM_disso_delta Rct-i.png


C:\Users\austi\AppData\Local\Temp\ipykernel_27728\3985480093.py:98: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_for_plot, labels=chip_labels)


Saved plot: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\100nM_disso_graphs\100nM_disso_Q.png


C:\Users\austi\AppData\Local\Temp\ipykernel_27728\3985480093.py:98: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_for_plot, labels=chip_labels)


Saved plot: c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia\100nM_disso_graphs\100nM_disso_n.png
Plots saved in c:\Users\austi\OneDrive - UC San Diego\Documents\AranLab\BioSensorAnalysisSrcCode\EIS_Analysis\Austin_EIS\MegaExcel\Processed Chips for PCAI1ia
